In [1]:
import numpy as np
import igraph as ig
from tqdm.auto import tqdm

import direct_social_belief_sim_cy_sparse as sim_cy


def belief_graph_to_matrix(belief_graph, weight_attr="weight", default_weight=1.0, zero_diagonal=True):
    if isinstance(belief_graph, np.ndarray):
        W = np.asarray(belief_graph, dtype=np.float64).copy()
    elif hasattr(belief_graph, "toarray"):
        W = np.asarray(belief_graph.toarray(), dtype=np.float64)
    elif isinstance(belief_graph, ig.Graph):
        n = belief_graph.vcount()
        W = np.zeros((n, n), dtype=np.float64)
        has_weight = weight_attr in belief_graph.es.attributes()

        for edge in belief_graph.es:
            i, j = edge.tuple
            w = float(edge[weight_attr]) if has_weight and edge[weight_attr] is not None else default_weight
            W[i, j] = w
            if not belief_graph.is_directed():
                W[j, i] = w
    else:
        raise TypeError("belief_graph must be a NumPy matrix, scipy sparse matrix, or igraph.Graph")

    if W.ndim != 2 or W.shape[0] != W.shape[1]:
        raise ValueError("belief graph/weights must be a square matrix")

    if zero_diagonal:
        np.fill_diagonal(W, 0.0)

    return W


def agent_graph_to_csr_tuple(agent_graph, weight_attr="weight", default_weight=1.0):
    if not isinstance(agent_graph, ig.Graph):
        raise TypeError("agent_graph must be an igraph.Graph")

    rows, cols, data = [], [], []
    has_weight = weight_attr in agent_graph.es.attributes()

    for edge in agent_graph.es:
        i, j = edge.tuple
        w = float(edge[weight_attr]) if has_weight and edge[weight_attr] is not None else default_weight

        rows.append(i)
        cols.append(j)
        data.append(w)

        if not agent_graph.is_directed():
            rows.append(j)
            cols.append(i)
            data.append(w)

    n = agent_graph.vcount()
    rows = np.asarray(rows, dtype=np.int32)
    cols = np.asarray(cols, dtype=np.int32)
    data = np.asarray(data, dtype=np.float64)

    order = np.lexsort((cols, rows))
    rows = rows[order]
    cols = cols[order]
    data = data[order]

    indptr = np.zeros(n + 1, dtype=np.int32)
    np.add.at(indptr, rows + 1, 1)
    indptr = np.cumsum(indptr).astype(np.int32)

    return indptr, cols, data, (n, n)


def run_sparse_simulation_with_snapshots(
    agent_graph,
    belief_graph,
    *,
    beta_internal,
    beta_external,
    number_of_steps,
    history_every,
    focal_beliefs=None,
    belief_attr="beliefs",
    allowed_values=None,
    normalize_neighbor_influence=False,
    seed=42,
    show_progress=True,
):
    belief_weights = belief_graph_to_matrix(belief_graph)

    initial_beliefs = np.asarray(agent_graph.vs[belief_attr], dtype=np.float64)
    if initial_beliefs.ndim != 2:
        raise ValueError(f'agent_graph.vs["{belief_attr}"] must contain belief vectors')

    n_agents, n_beliefs = initial_beliefs.shape
    if belief_weights.shape != (n_beliefs, n_beliefs):
        raise ValueError("belief_graph size must match belief vector length")

    agent_csr = agent_graph_to_csr_tuple(agent_graph)
    current_beliefs = initial_beliefs.copy()

    history = [
        {
            "step": 0,
            "beliefs": current_beliefs.copy(),
            "mean": float(current_beliefs.mean()),
        }
    ]

    seed_rng = np.random.default_rng(seed)
    starts = range(0, number_of_steps, history_every)

    if show_progress:
        starts = tqdm(starts, desc="Simulation snapshots")

    for start in starts:
        chunk_steps = min(history_every, number_of_steps - start)
        chunk_seed = None if seed is None else int(seed_rng.integers(1, 2**63 - 1))

        config = sim_cy.ExchangeConfig(
            beta_internal=beta_internal,
            beta_social=beta_external,
            focal_beliefs=focal_beliefs,
            n_steps=chunk_steps,
            seed=chunk_seed,
        )

        graph_chunk, _ = sim_cy.run_exchange_fast_sparse(
            belief_weights,
            agent_csr,
            allowed_values=allowed_values,
            initial_beliefs=current_beliefs,
            config=config,
            normalize_neighbor_influence=normalize_neighbor_influence,
            return_history=False,
            show_progress=False,
        )

        current_beliefs = np.asarray(graph_chunk.vs["beliefs"], dtype=np.float64)

        history.append(
            {
                "step": start + chunk_steps,
                "beliefs": current_beliefs.copy(),
                "mean": float(current_beliefs.mean()),
            }
        )

    final_graph = agent_graph.copy()
    final_graph.vs[belief_attr] = [row.tolist() for row in current_beliefs]

    return final_graph, history

In [46]:
import pickle
import numpy as np
import pandas as pd


class CompatUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        # pickle был сохранен с NumPy 2.x, а читаем в NumPy 1.26
        if module.startswith("numpy._core"):
            module = module.replace("numpy._core", "numpy.core", 1)

        # совместимость pandas StringDtype между версиями
        if module == "pandas" and name == "StringDtype":
            def make_string_dtype(storage=None, na_value=None):
                return pd.StringDtype(storage=storage)
            return make_string_dtype

        return super().find_class(module, name)


def read_belief_matrix_from_pickle(path):
    with open(path, "rb") as f:
        df = CompatUnpickler(f).load()

    belief_names = list(df.columns)
    belief_weights = df.to_numpy(dtype=np.float64)

    return belief_weights, belief_names, df

In [49]:
import numpy as np
from scipy import sparse

from boccaletti2007 import generate_boccaletti_graph, summary

n_agents = 2000
m = 3

g = generate_boccaletti_graph(
    n_nodes=n_agents,
    m=m,
    seed=42,
)

agent_adjacency = np.array(g.get_adjacency().data, dtype=np.float64)
agent_csr = sparse.csr_matrix(agent_adjacency)

belief_weights, belief_names, belief_df = read_belief_matrix_from_pickle(
    "correlation_net_uk_full.pkl"
)

belief_weights = belief_weights.astype(np.float64)
np.fill_diagonal(belief_weights, 0.0)

n_beliefs = belief_weights.shape[0]

k = 5
focal_beliefs = list(range(k))

rng = np.random.default_rng(42)

initial_beliefs = rng.choice(
    np.linspace(-1.0, 1.0, 7),
    size=(g.vcount(), n_beliefs),
)

g.vs["beliefs"] = initial_beliefs.tolist()

print(g.vcount(), g.ecount(), agent_csr.nnz)
print(n_beliefs)
print(focal_beliefs)
print([belief_names[i] for i in focal_beliefs])

2000 5994 11988
20
[0, 1, 2, 3, 4]
['left_right_identification', 'gender_inequality', 'anti_lgbt', 'euroscepticism', 'anti_immigration']


In [50]:
final_graph, history = run_sparse_simulation_with_snapshots(
    g,
    belief_weights,
    beta_internal=1.0,
    beta_external=1.5,
    number_of_steps=100_000,
    history_every=1_000,
    focal_beliefs=focal_beliefs,
    seed=42,
    show_progress=True,
)

final_beliefs = np.asarray(final_graph.vs["beliefs"])

print(final_beliefs.shape)
print(history[-1]["step"], history[-1]["mean"])

Simulation snapshots:   0%|          | 0/100 [00:00<?, ?it/s]

(2000, 20)
100000 0.02974166666666661


# Full simulation pipeline

In [59]:
import json
import itertools
from pathlib import Path
from datetime import datetime

import numpy as np
from tqdm.auto import tqdm


def run_beta_grid_experiment(
    agent_graph,
    belief_graph,
    param_grid,
    *,
    focal_beliefs,
    number_of_steps,
    history_every,
    results_root="results",
    experiment_name=None,
    seed=42,
    allowed_values=None,
    normalize_neighbor_influence=False,
    show_progress=True,
    overwrite=False,
    save_agent_graph=True,
    agent_graph_file="agent_graph.pkl",
):
    if "beta_internal" not in param_grid or "beta_external" not in param_grid:
        raise ValueError('param_grid must contain "beta_internal" and "beta_external"')

    focal_beliefs_saved = None if focal_beliefs is None else [int(x) for x in focal_beliefs]

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    if experiment_name is None:
        experiment_name = f"beta_grid_{timestamp}"

    results_root = Path(results_root)
    results_dir = results_root / experiment_name

    if results_dir.exists() and not overwrite:
        results_dir = results_root / f"{experiment_name}_{timestamp}"

    results_dir.mkdir(parents=True, exist_ok=overwrite)

    graph_path = results_dir / agent_graph_file if save_agent_graph else None
    if save_agent_graph:
        agent_graph.write_pickle(str(graph_path))

    beta_internal_values = list(param_grid["beta_internal"])
    beta_external_values = list(param_grid["beta_external"])

    combinations = list(
        itertools.product(
            beta_internal_values,
            beta_external_values,
        )
    )

    manifest = {
        "experiment_name": experiment_name,
        "created_at": timestamp,
        "results_dir": str(results_dir),
        "number_of_steps": int(number_of_steps),
        "history_every": int(history_every),
        "focal_beliefs": focal_beliefs_saved,
        "param_grid": {
            "beta_internal": beta_internal_values,
            "beta_external": beta_external_values,
        },
        "runs": [],
        "agent_graph_file": agent_graph_file if save_agent_graph else None,
        "agent_graph_path": str(graph_path) if graph_path is not None else None,
    }

    iterator = enumerate(combinations)
    if show_progress:
        iterator = tqdm(iterator, total=len(combinations), desc="Beta grid")

    for run_idx, (beta_internal, beta_external) in iterator:
        run_seed = None if seed is None else int(seed + run_idx)

        final_graph, history = run_sparse_simulation_with_snapshots(
            agent_graph,
            belief_graph,
            beta_internal=float(beta_internal),
            beta_external=float(beta_external),
            number_of_steps=int(number_of_steps),
            history_every=int(history_every),
            focal_beliefs=focal_beliefs,
            allowed_values=allowed_values,
            normalize_neighbor_influence=normalize_neighbor_influence,
            seed=run_seed,
            show_progress=False,
        )

        steps = np.asarray([item["step"] for item in history], dtype=np.int64)
        means = np.asarray([item["mean"] for item in history], dtype=np.float64)
        belief_history = np.stack([item["beliefs"] for item in history]).astype(np.float64)
        final_beliefs = np.asarray(final_graph.vs["beliefs"], dtype=np.float64)

        run_meta = {
            "run_idx": int(run_idx),
            "beta_internal": float(beta_internal),
            "beta_external": float(beta_external),
            "focal_beliefs": focal_beliefs_saved,
            "number_of_steps": int(number_of_steps),
            "history_every": int(history_every),
            "seed": run_seed,
            "n_snapshots": int(len(history)),
            "result_file": None,
        }

        file_name = (
            f"run_{run_idx:04d}"
            f"_bi_{float(beta_internal):.6g}"
            f"_be_{float(beta_external):.6g}"
            ".npz"
        )
        file_path = results_dir / file_name
        run_meta["result_file"] = file_name

        np.savez_compressed(
            file_path,
            steps=steps,
            means=means,
            belief_history=belief_history,
            final_beliefs=final_beliefs,
            focal_beliefs=np.asarray(
                [] if focal_beliefs_saved is None else focal_beliefs_saved,
                dtype=np.int64,
            ),
            beta_internal=np.asarray(float(beta_internal)),
            beta_external=np.asarray(float(beta_external)),
            seed=np.asarray(-1 if run_seed is None else run_seed, dtype=np.int64),
            metadata=json.dumps(run_meta, ensure_ascii=False),
        )

        manifest["runs"].append(run_meta)

    manifest_path = results_dir / "manifest.json"
    manifest["manifest_file"] = str(manifest_path)

    manifest_path.write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    return results_dir, manifest

In [62]:
param_grid = {
    "beta_internal": [0.5, 2.0],
    "beta_external": [0.5, 2.0],
}

ex_number = n_agents * len(belief_weights)
x = ex_number * 100


results_dir, manifest = run_beta_grid_experiment(
    g,
    belief_weights,
    param_grid,
    focal_beliefs=focal_beliefs,
    number_of_steps=x,
    history_every=ex_number,
    results_root="results",
    experiment_name="ex_1_beta_grid",
    seed=42,
)

print(results_dir)

Beta grid:   0%|          | 0/4 [00:00<?, ?it/s]

results\ex_1_beta_grid


# Sanity check

In [65]:
import json
import numpy as np

target = next(
    run for run in manifest["runs"]
    if run["beta_internal"] == 0.5 and run["beta_external"] == 2.0
)

file_path = results_dir / target["result_file"]

data = np.load(file_path, allow_pickle=False)

steps = data["steps"]
means = data["means"]
belief_history = data["belief_history"]
final_beliefs = data["final_beliefs"]
focal_beliefs_saved = data["focal_beliefs"].tolist()
metadata = json.loads(str(data["metadata"]))

print(metadata)
print("steps:", steps.shape)
print("belief_history:", belief_history.shape)
print("final_beliefs:", final_beliefs.shape)

{'run_idx': 1, 'beta_internal': 0.5, 'beta_external': 2.0, 'focal_beliefs': [0, 1, 2, 3, 4], 'number_of_steps': 4000000, 'history_every': 40000, 'seed': 43, 'n_snapshots': 101, 'result_file': 'run_0001_bi_0.5_be_2.npz'}
steps: (101,)
belief_history: (101, 2000, 20)
final_beliefs: (2000, 20)


In [67]:
belief_history.shape

(101, 2000, 20)

In [69]:
from pathlib import Path
import numpy as np

from boccaletti2007 import generate_boccaletti_graph
from polarisation import effective_dimensionality_from_graph


def load_experiment_graph(results_dir, file_name, snapshot="final"):
    path = Path(results_dir) / file_name

    with np.load(path, allow_pickle=False) as data:
        if snapshot == "final":
            beliefs = data["final_beliefs"]
        elif snapshot == "first":
            beliefs = data["belief_history"][0]
        elif snapshot == "last":
            beliefs = data["belief_history"][-1]
        elif isinstance(snapshot, int):
            beliefs = data["belief_history"][snapshot]
        else:
            raise ValueError('snapshot must be "final", "first", "last", or int')

    graph = generate_boccaletti_graph(
        n_nodes=beliefs.shape[0],
        m=3,
        seed=42,
    )
    graph.vs["beliefs"] = beliefs.tolist()
    return graph

In [87]:
import numpy as np
from pathlib import Path
from polarisation import effective_dimensionality_over_time

path = Path("results/ex_1_beta_grid") / "run_0002_bi_2_be_0.5.npz"

with np.load(path, allow_pickle=False) as data:
    belief_history = data["belief_history"]
    steps = data["steps"]

series = effective_dimensionality_over_time(
    belief_history,
    steps=steps,
    snapshot_indices=None,  # все snapshots
)

for step, d_eff, pc1 in zip(series.steps, series.d_eff, series.pc1_share):
    print(step, d_eff, pc1)

0 19.950224194959 0.05643926801483823
40000 17.494916734816588 0.18894137942230535
80000 12.853397153806435 0.34511397815278677
120000 9.312075596595044 0.46851305834886786
160000 7.182819361558273 0.5535692725368392
200000 5.960787195924105 0.6088502705068128
240000 5.161997158754988 0.6493541171281567
280000 4.658476204342198 0.6763168183139345
320000 4.269231862744211 0.6981832134418791
360000 4.020667466694161 0.7125991460428589
400000 3.892865577165401 0.7198453730500488
440000 3.806451193703557 0.7250175275446428
480000 3.7439919157990187 0.7291830401617608
520000 3.687547373178772 0.7334409209649914
560000 3.638255747838882 0.7362993959526707
600000 3.6683529923800555 0.733846117726439
640000 3.67878570008178 0.7330405155705781
680000 3.6643252598075167 0.7341147273898101
720000 3.7107364882856073 0.7309143651884796
760000 3.6555952487289787 0.7351201614398529
800000 3.6394489789658646 0.736463932410909
840000 3.6314794747563304 0.7370032497142087
880000 3.6590624015311057 0.734

In [1]:
from polarisation import save_effective_dimensionality_history_by_params

dim_by_params = save_effective_dimensionality_history_by_params(
    "results/ex_1_beta_grid",
    "results/ex_1_beta_grid/effective_dimensionality_by_params.json",
)